# Data Cleaning & Preprocessing

### Import Required Libraries

In [29]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px

### Load Dataset

In [30]:
df = pd.read_excel("../data/raw/indian-job-market-dataset-2025.xlsx")

### Create Working Copy

In [31]:
df_clean = df.copy()

### Duplicate Record Verification

In [32]:
df_clean.duplicated().sum()

np.int64(247)

### Handle Missing Values

Missing values were handled using statistical imputation techniques. For numerical attributes, the choice between mean and median was based on the distribution of the data. Columns with approximately symmetric distributions were imputed using the mean, while skewed distributions were imputed using the median to reduce the influence of outliers.

For categorical attributes, missing values were replaced using the mode (most frequently occurring value), ensuring that all missing entries were filled while preserving the categorical nature of the data.

#### Calculate Mean and Median for Numerical Data

In [33]:
numeric_columns = df_clean.select_dtypes(include=["number"]).columns

statistics = pd.DataFrame({
    "Mean": df_clean[numeric_columns].mean(),
    "Median": df_clean[numeric_columns].median(),
    "Missing Values": df_clean[numeric_columns].isnull().sum()
})

statistics

,Mean,Median,Missing Values
jobId,2.069854e+11,2.508250e+11,0
companyId,2.784642e+07,2.892790e+06,0
ReviewsCount,1.466259e+04,9.620000e+02,35252
AggregateRating,3.679048e+00,3.700000e+00,35252
minimumSalary,1.983092e+05,0.000000e+00,571
maximumSalary,3.186096e+05,0.000000e+00,571
minimumExperience,3.957744e+00,3.000000e+00,571
maximumExperience,7.732647e+00,7.000000e+00,571


### Fill Missing Numerical Values

In [34]:
df_clean["minimumSalary"] = df_clean["minimumSalary"].fillna(
    df_clean["minimumSalary"].median()
)

df_clean["maximumSalary"] = df_clean["maximumSalary"].fillna(
    df_clean["maximumSalary"].median()
)

df_clean["minimumExperience"] = df_clean["minimumExperience"].fillna(
    df_clean["minimumExperience"].median()
)

df_clean["maximumExperience"] = df_clean["maximumExperience"].fillna(
    df_clean["maximumExperience"].median()
)

df_clean["ReviewsCount"] = df_clean["ReviewsCount"].fillna(
    df_clean["ReviewsCount"].median()
)

df_clean["AggregateRating"] = df_clean["AggregateRating"].fillna(
    df_clean["AggregateRating"].mean()
)

#### Calculate Mode for Categorical Data

In [35]:
categorical_mode = pd.DataFrame({
    "Mode": df_clean.select_dtypes(include=["object", "string"]).mode().iloc[0]
})

categorical_mode

,Mode
title,Application Developer
currency,INR
jobUploaded,4 Days Ago
companyName,Accenture
tagsAndSkills,"Retail,Lead generation,Forex,Investment produc..."
experience,5-10 Yrs
salary,Not disclosed
location,Bengaluru
jobDescription,Disclaimer: This job description has been sour...


### Fill Missing Categorical Values

In [36]:
df_clean["companyName"] = df_clean["companyName"].fillna(
    df_clean["companyName"].mode()[0]
)

df_clean["tagsAndSkills"] = df_clean["tagsAndSkills"].fillna(
    df_clean["tagsAndSkills"].mode()[0]
)

df_clean["experience"] = df_clean["experience"].fillna(
    df_clean["experience"].mode()[0]
)

### Standardize Text Columns

Text-based attributes are standardized to improve consistency throughout the dataset. This preprocessing step removes unnecessary whitespace and ensures a uniform format, reducing inconsistencies that may affect grouping, filtering, and analytical results.

In [37]:
text_columns = df_clean.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

### Remove Extra Spaces Between Words

In [38]:
for col in text_columns:
    df_clean[col] = df_clean[col].str.replace(r"\s+", " ", regex=True)

### Verify Standardization

In [39]:
space_counts = {
    col: (df_clean[col] != df_clean[col].str.strip()).sum()
    for col in text_columns
}

pd.DataFrame(
    space_counts.items(),
    columns=["Column", "Remaining Leading/Trailing Spaces"]
)

,Column,Remaining Leading/Trailing Spaces
0,title,0
1,currency,0
2,jobUploaded,0
3,companyName,0
4,tagsAndSkills,0
5,experience,0
6,salary,0
7,location,0
8,jobDescription,0


### Create Average Salary Column

In [40]:
df_clean["averageSalary"] = (
    df_clean["minimumSalary"] + df_clean["maximumSalary"]
) / 2

### Verify Salary Columns

In [41]:
df_clean[
    ["minimumSalary", "maximumSalary", "averageSalary"]
].head()

,minimumSalary,maximumSalary,averageSalary
0,200000.0,400000.0,300000.0
1,300000.0,500000.0,400000.0
2,0.0,0.0,0.0
3,70000.0,200000.0,135000.0
4,800000.0,1800000.0,1300000.0


### Create Average Experience Column

In [42]:
df_clean["averageExperience"] = (
    df_clean["minimumExperience"] + df_clean["maximumExperience"]
) / 2

### Verify Experience Columns

In [43]:
df_clean[
    [
        "minimumExperience",
        "maximumExperience",
        "averageExperience"
    ]
].head()

,minimumExperience,maximumExperience,averageExperience
0,2.0,4.0,3.0
1,6.0,11.0,8.5
2,12.0,18.0,15.0
3,0.0,3.0,1.5
4,5.0,10.0,7.5


### Final Data Validation

In [44]:
df_clean.isnull().sum()

title                0
jobId                0
currency             0
jobUploaded          0
companyName          0
tagsAndSkills        0
experience           0
salary               0
location             0
companyId            0
ReviewsCount         0
AggregateRating      0
jobDescription       0
minimumSalary        0
maximumSalary        0
minimumExperience    0
maximumExperience    0
averageSalary        0
averageExperience    0
dtype: int64

In [45]:
df_clean.shape

(97929, 19)

In [46]:
df_clean[["averageSalary", "averageExperience"]].head()

,averageSalary,averageExperience
0,300000.0,3.0
1,400000.0,8.5
2,0.0,15.0
3,135000.0,1.5
4,1300000.0,7.5


In [47]:
df_clean.sample(5, random_state=42)

,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience,averageSalary,averageExperience
84176,Zonal Manager - Affordable Housing,260925918114,INR,7 Days Ago,Bajaj Finance,"Sales,Databases,Development,Talent Management,...",14-15 Yrs,Not disclosed,Pune,122,8015.0,3.900000,Required Qualifications and Experience <br><br...,0.0,0.0,14.0,15.0,0.0,14.5
39838,Management trainee collections,280825011954,INR,8 Days Ago,Genpact,"joining formalities,accounts receivable,orient...",3-7 Yrs,Not disclosed,Jaipur,30975,38095.0,3.700000,<p><strong>Ready to shape the future of work? ...,0.0,0.0,3.0,7.0,0.0,5.0
77820,Senior C # Developer,300925029858,INR,3 Days Ago,Katapie Consulting Services,"C Hash,Motion Control,Opencv,Image Processing,...",6-9 Yrs,20-30 Lacs PA,Remote,125133963,962.0,3.679048,"Requires expertise in C# & Python , handling m...",2000000.0,3000000.0,6.0,9.0,2500000.0,7.5
68128,Machine Shop Electrical Maintenance,240925506188,INR,9 Days Ago,Happy Forging,"Electrical drawing,Air compressor,Basic,PLC,Pr...",4-7 Yrs,Not disclosed,Ludhiana,124534690,136.0,3.700000,<li> <span> <i> </i> </span> <span> Troublesho...,0.0,0.0,4.0,7.0,0.0,5.5
63223,Store Manager,300925030778,INR,2 Days Ago,Suvidha Placements,"Stores,Store Operations,consumable,Inventory M...",12-16 Yrs,5-7 Lacs PA,"Barjora, Durgapur",85003,962.0,3.679048,Managing stores of consumable and store items ...,500000.0,700000.0,12.0,16.0,600000.0,14.0


### Export Clean Dataset

In [48]:
df_clean.to_csv(
    "../data/processed/cleaned_job_market.csv",
    index=False
)

## Data Cleaning Summary

### Preprocessing Operations Performed

- Verified the dataset for duplicate records.
- Handled missing values using appropriate statistical imputation techniques (Mean, Median, and Mode) based on the characteristics of each attribute.
- Standardized text-based columns by removing leading/trailing whitespace and normalizing text formatting to improve consistency.
- Engineered new analytical features:
  - `averageSalary`
  - `averageExperience`
- Validated the processed dataset to ensure data completeness, consistency, and readiness for analysis.
- Exported the final cleaned dataset for subsequent stages of the project.

### Outcome

The dataset has been transformed into a clean, consistent, and analysis-ready resource. The preprocessing pipeline improves data quality, enhances feature usability, and establishes a reliable foundation for exploratory data analysis, interactive visualizations, dashboard development, and advanced analytical tasks. The engineered features further simplify downstream analysis and enable more meaningful insights into salary trends, experience requirements, hiring patterns, and the Indian job market.